In [1]:
import fitz
import re
from pdfminer.pdfpage import PDFPage

class PDFService:
    def __init__(self, min_text_threshold=50, min_text_ratio=0.1, max_image_coverage=0.6):
        """
        Inicializa o detector de tipo de PDF

        Args:
            pdf_file: Arquivo PDF a ser analisado
            min_text_threshold: Número mínimo de caracteres para considerar como texto
            min_text_ratio: Proporção mínima de texto em relação ao conteúdo total
            max_image_coverage: Cobertura máxima de imagem (0.6 = 60%) antes de considerar como PDF de imagem
        """
        self.min_text_threshold = min_text_threshold
        self.min_text_ratio = min_text_ratio
        self.max_image_coverage = max_image_coverage


    def __calculate_image_coverage(self, page, page_area):
        """
        Calcula a porcentagem da página ocupada por imagens

        Args:
            page: Objeto da página do PyMuPDF
            page_area: Área total da página

        Returns:
            float: Porcentagem de cobertura de imagem (0.0 a 1.0)
        """
        image_list = page.get_images()
        total_image_area = 0

        for img_index, img in enumerate(image_list):
            try:
                # Obtém informações da imagem
                xref = img[0]

                # Tenta obter o bbox da imagem através dos objetos de página
                # Busca por todas as ocorrências desta imagem na página
                image_instances = []

                # Método 1: Usar get_image_bbox se disponível
                try:
                    bbox = page.get_image_bbox(img)
                    if bbox and bbox != fitz.Rect():
                        image_instances.append(bbox)
                except:
                    pass

                # Método 2: Procurar através do conteúdo da página
                if not image_instances:
                    try:
                        # Analisa o conteúdo da página para encontrar transformações de imagem
                        page_dict = page.get_text("dict")

                        # Procura por blocos de imagem
                        if 'blocks' in page_dict:
                            for block in page_dict['blocks']:
                                if block.get('type') == 1:  # Tipo 1 = bloco de imagem
                                    bbox = fitz.Rect(block['bbox'])
                                    image_instances.append(bbox)
                    except:
                        pass

                # Método 3: Estimativa baseada em dimensões padrão se não encontrar bbox
                if not image_instances:
                    try:
                        # Obtém dimensões da imagem
                        pix = fitz.Pixmap(page.parent, xref)
                        if pix:
                            # Estima posição baseada no tamanho da imagem vs página
                            img_width = pix.width
                            img_height = pix.height

                            # Calcula escala aproximada (assumindo DPI padrão)
                            scale_x = min(page.rect.width * 0.8, img_width) / img_width
                            scale_y = min(page.rect.height * 0.8, img_height) / img_height
                            scale = min(scale_x, scale_y)

                            estimated_width = img_width * scale
                            estimated_height = img_height * scale

                            # Cria bbox estimado (centralizado)
                            x0 = (page.rect.width - estimated_width) / 2
                            y0 = (page.rect.height - estimated_height) / 2
                            x1 = x0 + estimated_width
                            y1 = y0 + estimated_height

                            estimated_bbox = fitz.Rect(x0, y0, x1, y1)
                            image_instances.append(estimated_bbox)

                            pix = None  # Libera memória
                    except:
                        # Se tudo falhar, assume uma área padrão conservadora
                        default_area = min(page_area * 0.3, page_area)  # Máximo 30% da página
                        total_image_area += default_area
                        continue

                # Soma as áreas de todas as instâncias desta imagem
                for bbox in image_instances:
                    area = bbox.width * bbox.height
                    if area > 0:
                        total_image_area += area

            except Exception:
                # Em caso de erro, assume área conservadora
                default_area = min(page_area * 0.2, page_area * 0.2)
                total_image_area += default_area
                continue

        # Calcula a porcentagem, limitando a 100%
        coverage = min(total_image_area / page_area, 1.0) if page_area > 0 else 0
        return coverage

    def detect_pdf_type(self, pdf_file) -> str:
        """
        Detecta se o PDF contém texto extraível ou é baseado em imagens
        """
        try:
            with open(pdf_file, 'rb') as f:
                text = False

                for page in PDFPage.get_pages(f):
                    if "Font" in page.resources.keys():
                        text = True
                        break

            if text is False:
                pdf_type = "IMAGE"
                return pdf_type

            doc = fitz.open("../data/docs-test/CarteiraHabilitacao_Bruno.pdf")

            total_pages = len(doc)
            text_pages = 0
            image_pages = 0
            total_text_length = 0
            total_image_coverage = 0
            pages_analysis = []

            for page_num in range(total_pages):
                page = doc[page_num]
                page_area = page.rect.width * page.rect.height

                # Extrai texto da página
                text = page.get_text().strip()
                clean_text = re.sub(r'\s+', ' ', text)

                # Calcula cobertura de imagem
                image_coverage = self.__calculate_image_coverage(page, page_area)
                total_image_coverage += image_coverage

                # Conta imagens na página
                image_list = page.get_images()

                # Análise aprimorada da página
                has_meaningful_text = len(clean_text) > self.min_text_threshold
                has_high_image_coverage = image_coverage > self.max_image_coverage
                has_images = len(image_list) > 0

                # Lógica de classificação aprimorada
                if has_meaningful_text and not has_high_image_coverage:
                    page_type = "TEXT"
                    text_pages += 1
                    total_text_length += len(clean_text)
                elif has_high_image_coverage or (has_images and not has_meaningful_text):
                    page_type = "IMAGE"
                    image_pages += 1
                    total_text_length += len(clean_text)
                elif has_meaningful_text and has_images:
                    page_type = "TEXT"
                    text_pages += 1
                    total_text_length += len(clean_text)
                else:
                    page_type = "UNKNOWN"
                    total_text_length += len(clean_text)

                page_analysis = {
                    'page_number': page_num + 1,
                    'type': page_type,
                    'text_length': len(clean_text),
                    'image_count': len(image_list),
                    'image_coverage': round(image_coverage, 3),
                    'has_meaningful_text': has_meaningful_text,
                    'has_high_image_coverage': has_high_image_coverage
                }

                pages_analysis.append(page_analysis)

            doc.close()

            if text_pages > image_pages:
                pdf_type = "TEXT"
            else:
                pdf_type = "IMAGE"

            return pdf_type

        except Exception:
            pdf_type = "UNKNOWN"
            return pdf_type

    def extract_pdf_text(self, pdf_file):
        """
        Extrai texto de um PDF mantendo a estrutura original.
        """
        try:
            pdf_doc = fitz.open(pdf_file)
            total_pages = len(pdf_doc)
            full_text = ""

            for page_num in range(total_pages):
                page = pdf_doc.load_page(page_num)
                text_blocks = page.get_text("blocks")
                page_text = ""
                for block in text_blocks:
                    page_text += block[4]
                full_text += f"\n" + page_text
            pdf_text = full_text.strip()
            pdf_doc.close()

            return pdf_text
        except Exception:
            return ""

if __name__ == "__main__":
    pdf_path = "../data/docs-test/ComprovanteResidencia_Bruno.pdf"
    pdf_service = PDFService()
    pdf_type = pdf_service.detect_pdf_type(pdf_path)
    print(f"Tipo do PDF: {pdf_type}")

    pdf_text = pdf_service.extract_pdf_text(pdf_path)
    print(f"Texto extraído do PDF:\n{pdf_text[:500]}")  # Exibe os primeiros 500 caracteres

Tipo do PDF: IMAGE
Texto extraído do PDF:
Valores Faturados
Itens da Fatura
Unid.
Quant.
Preço Unit
Valor (R$)
 PIS/COFINS
Base Calc.
Aliq.
ICMS
Tarifa Unit.
ICMS
ICMS
Energia Elétrica
kWh
     114 
 1,01703171 
       115,92
          3,90
        115,92           18,00          20,86
 0,79969000 
Contrib Ilum Publica Municipal
         24,19
Multa 2% sobre conta de 02/2025
          2,47
Juros 1%am sobre conta 01/25 pg 06/03/25
          1,13
Correção IPCA/IGPM s/ conta 01/25 pg 06/03/25
          0,19
TOTAL
        143,90           3
